# Fairness, Bias, and Explainability (SHAP/LIME)
This notebook demonstrates: feature importances, SHAP explanations (global + local), bias checks across sensitive groups, and practical mitigation steps.

**Dataset**: Synthetic tabular classification dataset with sensitive attributes: `gender`, `age_group`, `race`.

**Model**: Logistic Regression (baseline) and a variant trained without sensitive features.

> If SHAP is not installed, install dependencies by running the install cell below.

In [ ]:
# Install dependencies (run if needed)
# !pip install shap lime scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    roc_auc_score, accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
)
from sklearn.inspection import permutation_importance

import shap
import warnings
warnings.filterwarnings('ignore')

RANDOM_STATE = 42

## 1) Create a synthetic dataset with sensitive groups
We create a dataset where the target depends on non-sensitive features **and** has group-dependent biases (to illustrate measurable fairness issues).

In [ ]:
def make_synthetic(n=5000, seed=42):
    rng = np.random.default_rng(seed)

    # Sensitive attributes
    gender = rng.choice(['M', 'F'], size=n, p=[0.52, 0.48])
    age_group = rng.choice(['young', 'mid', 'old'], size=n, p=[0.38, 0.37, 0.25])
    race = rng.choice(['A', 'B', 'C'], size=n, p=[0.45, 0.35, 0.20])

    # Non-sensitive numeric features
    x1 = rng.normal(loc=0.0, scale=1.0, size=n)
    x2 = rng.normal(loc=1.0, scale=1.5, size=n)
    x3 = rng.uniform(-2, 2, size=n)

    # A background bias term: group-dependent offsets
    # (These offsets are intentionally chosen so that groups get different base risk/label rates.)
    group_offset = (
        (gender == 'F') * -0.35 +
        (age_group == 'old') * 0.30 +
        (age_group == 'young') * -0.10 +
        (race == 'B') * 0.25 +
        (race == 'C') * -0.20
    )

    # True signal
    logit = (
        1.1 * x1
        + 0.7 * x2
        - 0.9 * x3
        + 0.8 * (age_group == 'mid')
        + group_offset
    )

    # Convert to probabilities and sample label
    p = 1 / (1 + np.exp(-logit))
    y = rng.binomial(1, p, size=n)

    df = pd.DataFrame({
        'gender': gender,
        'age_group': age_group,
        'race': race,
        'x1': x1,
        'x2': x2,
        'x3': x3,
        'label': y
    })
    return df

df = make_synthetic(n=6000, seed=RANDOM_STATE)
df.head()

In [ ]:
df['label'].value_counts(normalize=True)

## 2) Train baseline model (includes sensitive features)
We one-hot encode categorical columns and train a logistic regression model.

In [ ]:
sensitive_features = ['gender', 'age_group', 'race']
categorical_features = sensitive_features
numeric_features = ['x1', 'x2', 'x3']
target = 'label'

X = df.drop(columns=[target])
y = df[target].values

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=RANDOM_STATE, stratify=y
)

preprocess = ColumnTransformer(
    transformers=[
        ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features),
        ('num', 'passthrough', numeric_features),
    ]
)

baseline_model = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)

baseline = Pipeline(steps=[('preprocess', preprocess), ('model', baseline_model)])
baseline.fit(X_train, y_train)

proba_test = baseline.predict_proba(X_test)[:, 1]
pred_test = (proba_test >= 0.5).astype(int)

metrics = {
    'roc_auc': roc_auc_score(y_test, proba_test),
    'accuracy': accuracy_score(y_test, pred_test),
    'precision': precision_score(y_test, pred_test),
    'recall': recall_score(y_test, pred_test),
    'f1': f1_score(y_test, pred_test),
}
metrics

## 3) Feature importances
We compute two types of importances:
1. **Permutation importance** on the test set
2. **Coefficient-based importance** from the logistic regression (after one-hot encoding)

In [ ]:
# --- Permutation importance ---
perm = permutation_importance(baseline, X_test, y_test, n_repeats=10, random_state=RANDOM_STATE, scoring='roc_auc')
sorted_idx = perm.importances_mean.argsort()[::-1]
feature_names = perm.feature_names if hasattr(perm, 'feature_names') else None

# In sklearn versions where perm.feature_names is unavailable, reconstruct names from encoder + numeric
if feature_names is None:
    ohe = baseline.named_steps['preprocess'].named_transformers_['cat']
    cat_names = ohe.get_feature_names_out(categorical_features)
    feature_names = list(cat_names) + numeric_features

imp_df = pd.DataFrame({
    'feature': feature_names,
    'perm_importance_mean': perm.importances_mean,
    'perm_importance_std': perm.importances_std,
}).iloc[sorted_idx].reset_index(drop=True)

plt.figure(figsize=(10, 6))
sns.barplot(data=imp_df.head(15), y='feature', x='perm_importance_mean', color='#4C72B0')
plt.title('Top 15 Permutation Importances (ROC-AUC)')
plt.tight_layout()
plt.show()

# --- Coefficient importance (logistic regression weights) ---
# Get one-hot expanded feature names
ohe = baseline.named_steps['preprocess'].named_transformers_['cat']
cat_names = ohe.get_feature_names_out(categorical_features)
expanded_feature_names = list(cat_names) + numeric_features

coef = baseline.named_steps['model'].coef_.reshape(-1)
coef_abs = np.abs(coef)
coef_df = pd.DataFrame({
    'feature': expanded_feature_names,
    'abs_coefficient': coef_abs
}).sort_values('abs_coefficient', ascending=False)

plt.figure(figsize=(10, 6))
sns.barplot(data=coef_df.head(15), y='feature', x='abs_coefficient', color='#55A868')
plt.title('Top 15 Coefficient Magnitudes (Logistic Regression)')
plt.tight_layout()
plt.show()
coef_df.head(10)

## 4) SHAP explanations (global + local)
Logistic regression is linear; SHAP can be computed using `shap.LinearExplainer` / `shap.Explainer`.
We will compute SHAP values on a transformed (one-hot + numeric) representation.

In [ ]:
# Transform train/test data to the model's feature space
X_train_t = baseline.named_steps['preprocess'].transform(X_train)
X_test_t = baseline.named_steps['preprocess'].transform(X_test)

# Wrap explainer: use LinearExplainer for linear model
model = baseline.named_steps['model']

explainer = shap.LinearExplainer(model, X_train_t, feature_perturbation='interventional')

# Compute SHAP for a subset for speed
n_explain = min(1000, X_test_t.shape[0])
idx = np.random.RandomState(RANDOM_STATE).choice(X_test_t.shape[0], n_explain, replace=False)
X_explain_t = X_test_t[idx]
X_explain_raw = X_test.iloc[idx].reset_index(drop=True)

shap_values = explainer.shap_values(X_explain_t)

feature_names = expanded_feature_names
shap_values = np.array(shap_values)  # shape: (n_samples, n_features)

# Create SHAP summary plot (global importance)
shap.summary_plot(shap_values, features=X_explain_t, feature_names=feature_names, show=False)
plt.tight_layout()
plt.show()

# Dependence plot for top feature by mean |SHAP|
mean_abs = np.abs(shap_values).mean(axis=0)
top_feat_idx = mean_abs.argmax()
top_feat_name = feature_names[top_feat_idx]

shap.dependence_plot(top_feat_idx, shap_values, X_explain_t, feature_names=feature_names, show=False)
plt.tight_layout()
plt.show()
top_feat_name

### Local explanation example
We pick one test sample and show SHAP force plot.

In [ ]:
# Choose one instance to explain
i = 0
x_raw = X_explain_raw.iloc[i]
x_t = X_explain_t[i:i+1]

pred = baseline.predict_proba(pd.DataFrame([x_raw]))[:, 1][0]
shap_i = shap_values[i]

# Force plot (works best in notebook; matplotlib output)
shap.initjs()
force = shap.force_plot(explainer.expected_value, shap_i, x_t, feature_names=feature_names, matplotlib=True, show=False)
plt.tight_layout()
plt.show()

print('Instance to explain:')
print(x_raw)
print(f'Predicted probability of label=1: {pred:.3f}')

## 5) Bias / fairness checks across sensitive groups
We compute:
- **Selection rate** = fraction predicted as positive
- **TPR (Equal Opportunity)** = P(ŷ=1 | y=1)
We report differences relative to a reference group for interpretability.

In [ ]:
def group_metrics(df_test, y_true, y_pred, sensitive_col):
    out = []
    for g, gdf in df_test.groupby(sensitive_col):
        idx = gdf.index
        yt = y_true[idx]
        yp = y_pred[idx]
        selection_rate = yp.mean()
        # TPR only meaningful when there are positives
        if yt.sum() == 0:
            tpr = np.nan
        else:
            tp = ((yp == 1) & (yt == 1)).sum()
            tpr = tp / yt.sum()
        out.append({
            sensitive_col: g,
            'count': len(yt),
            'selection_rate': selection_rate,
            'tpr_equal_opportunity': tpr,
            'precision': precision_score(yt, yp, zero_division=0),
            'recall': recall_score(yt, yp, zero_division=0),
            'f1': f1_score(yt, yp, zero_division=0),
        })
    return pd.DataFrame(out)

# Prepare test frame for grouping
df_test = X_test.copy()
df_test['label'] = y_test
df_test['pred'] = pred_test

bias_reports = {}
for s in sensitive_features:
    bias_reports[s] = group_metrics(df_test, y_test, pred_test, s)
    print('
=== Bias report for', s, '===')
    display(bias_reports[s].sort_values('selection_rate', ascending=False).reset_index(drop=True))

In [ ]:
# Visualize selection rate and TPR gaps
fig, axes = plt.subplots(1, len(sensitive_features), figsize=(14, 4), sharey=False)
for ax, s in zip(axes, sensitive_features):
    rep = bias_reports[s].copy()
    sns.barplot(data=rep, x=s, y='selection_rate', ax=ax, color='#C44E52')
    ax.set_title(f'Selection rate by {s}')
    ax.set_xlabel('')
    ax.set_ylabel('Selection rate')
plt.tight_layout()
plt.show()

fig, axes = plt.subplots(1, len(sensitive_features), figsize=(14, 4), sharey=False)
for ax, s in zip(axes, sensitive_features):
    rep = bias_reports[s].copy()
    sns.barplot(data=rep, x=s, y='tpr_equal_opportunity', ax=ax, color='#8172B3')
    ax.set_title(f'TPR (Equal Opportunity) by {s}')
    ax.set_xlabel('')
    ax.set_ylabel('TPR')
plt.tight_layout()
plt.show()

## 6) Mitigation 1: Remove sensitive features (train a variant)
We retrain the model using only numeric features (`x1`, `x2`, `x3`).
This is a simple practical mitigation: it removes direct sensitive information, reducing group-specific artifacts.

In [ ]:
numeric_only_features = ['x1', 'x2', 'x3']
preprocess_num = ColumnTransformer(
    transformers=[('num', 'passthrough', numeric_only_features)]
)

variant_model = LogisticRegression(max_iter=2000, class_weight='balanced', random_state=RANDOM_STATE)
variant = Pipeline(steps=[('preprocess', preprocess_num), ('model', variant_model)])
variant.fit(X_train, y_train)

proba_test_v = variant.predict_proba(X_test)[:, 1]
pred_test_v = (proba_test_v >= 0.5).astype(int)

variant_metrics = {
    'roc_auc': roc_auc_score(y_test, proba_test_v),
    'accuracy': accuracy_score(y_test, pred_test_v),
    'precision': precision_score(y_test, pred_test_v),
    'recall': recall_score(y_test, pred_test_v),
    'f1': f1_score(y_test, pred_test_v),
}
variant_metrics

In [ ]:
# Bias reports for variant
df_test_v = X_test.copy()
df_test_v['label'] = y_test
df_test_v['pred'] = pred_test_v

variant_bias_reports = {}
for s in sensitive_features:
    variant_bias_reports[s] = group_metrics(df_test_v, y_test, pred_test_v, s)
    print('
=== Variant bias report for', s, '===')
    display(variant_bias_reports[s].sort_values('selection_rate', ascending=False).reset_index(drop=True))

In [ ]:
# Compare Equal Opportunity (TPR) gap: max-min
def tpr_gap(rep):
    vals = rep['tpr_equal_opportunity'].dropna().values
    return float(np.max(vals) - np.min(vals)) if len(vals) > 1 else np.nan

for s in sensitive_features:
    base_gap = tpr_gap(bias_reports[s])
    var_gap = tpr_gap(variant_bias_reports[s])
    print(f'{s}: TPR gap baseline={base_gap:.3f} | variant={var_gap:.3f}')

## 7) Mitigation 2: Group-wise threshold adjustment
We apply **post-processing** to improve equal opportunity across a chosen sensitive attribute.

For each group of a selected attribute, we search thresholds that bring TPR closer to a target group's TPR.

This is a common practical technique when training-time mitigation alone is insufficient.

In [ ]:
def threshold_search_equal_opportunity(df_test, y_true, proba, sensitive_col, grid=None, target_group=None):
    if grid is None:
        grid = np.linspace(0.05, 0.95, 37)

    groups = sorted(df_test[sensitive_col].unique())
    if target_group is None:
        target_group = groups[0]

    # Compute baseline TPR per group at threshold 0.5
    baseline_pred = (proba >= 0.5).astype(int)

    def tpr_for_group(g, thr):
        idx = df_test.index[df_test[sensitive_col] == g].to_numpy()
        yt = y_true[idx]
        yp = (proba[idx] >= thr).astype(int)
        if yt.sum() == 0:
            return np.nan
        tp = ((yp == 1) & (yt == 1)).sum()
        return tp / yt.sum()

    target_tpr = tpr_for_group(target_group, 0.5)

    chosen = {}
    for g in groups:
        best_thr = 0.5
        best_dist = np.inf
        for thr in grid:
            tpr = tpr_for_group(g, thr)
            if np.isnan(tpr):
                continue
            dist = abs(tpr - target_tpr)
            if dist < best_dist:
                best_dist = dist
                best_thr = float(thr)
        chosen[g] = best_thr

    # Apply thresholds per group
    pred_adj = np.zeros_like(y_true)
    for g, thr in chosen.items():
        idx = df_test.index[df_test[sensitive_col] == g].to_numpy()
        pred_adj[idx] = (proba[idx] >= thr).astype(int)

    return chosen, pred_adj, target_tpr

# Choose attribute to equalize (you can change this)
attribute_to_mitigate = 'gender'

thresholds, pred_adj, target_tpr = threshold_search_equal_opportunity(
    df_test, y_test, proba_test, sensitive_col=attribute_to_mitigate, target_group=df_test[attribute_to_mitigate].mode()[0]
)
print('Chosen thresholds per group (baseline model):', thresholds)
print('Target group baseline TPR @0.5:', target_tpr)

In [ ]:
# Bias report after threshold adjustment
df_test_adj = X_test.copy()
df_test_adj['label'] = y_test
df_test_adj['pred'] = pred_adj

adj_reports = {}
for s in sensitive_features:
    adj_reports[s] = group_metrics(df_test_adj, y_test, pred_adj, s)

print('=== Threshold-adjusted bias report for', attribute_to_mitigate, '===')
display(adj_reports[attribute_to_mitigate].sort_values('selection_rate', ascending=False).reset_index(drop=True))

# Compare TPR gap
for s in sensitive_features:
    base_gap = tpr_gap(bias_reports[s])
    adj_gap = tpr_gap(adj_reports[s])
    print(f'{s}: TPR gap baseline={base_gap:.3f} | threshold-adjusted={adj_gap:.3f}')

## 8) Local explanations on baseline predictions (optional LIME note)
Requirement mentions SHAP or LIME. This notebook uses SHAP for local and global explanations.

(If you want LIME plots, you can add them later. For many tabular pipelines, SHAP is usually simpler.)

## Summary
- Baseline model (with sensitive attributes) shows measurable group differences (selection rate and equal opportunity/TPR gaps).
- Mitigation strategy 1 (removing sensitive attributes at training) can reduce gaps, at possible cost to some metrics.
- Mitigation strategy 2 (group-wise thresholding) further reduces TPR gaps via post-processing.
- SHAP identifies which one-hot expanded features and numeric inputs most influence predictions, enabling targeted scrutiny.